# Optimizer profile: overview
Peak GPU memory and steady-state step time for every baseline and every Sven variant, per architecture, at the
training set point (`study == "methods"`). Produced by `experiments/optimizer_profile.py` on an exclusively reserved node.

* **Sven (Gram, hooks)**: per-layer kernel, no Jacobian. Not possible for the batch-statistics ResNet.
* **Sven (Gram, full $J$)**: one `jacrev`, the $B\times P$ Jacobian, its Gram matrix, `eigh`.
* **Sven (Gram, chunked)**: the same in parameter groups of $f\cdot P$ (here $f=0.25$).
* **Sven (classic, rand. SVD)**: the $B\times P$ Jacobian and a randomized-SVD pseudo-inverse (`randomized_v2`).

In [ ]:
import sys
sys.path.insert(0, '.')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from style import set_style
import profile_helpers as ph
set_style()
PLOT_DIR = 'plots_v2/profile'
df = ph.load_profiles()          # ../profile_results_v2/*/*.json -> one tidy row per configuration
ARCHS = [a for a in ph.ARCH_ORDER if a in set(df.arch)]
df.groupby(['arch', 'study']).size().unstack(fill_value=0)

### Measurement quality
Step times are the **steady-state mean**: the first 20% of measured steps and any >3 MAD spike are dropped.
The panel below is the p90/p10 ratio of the raw per-step times for every configuration; anything above 1.3 is listed.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
unsteady = ph.plot_steadiness(df, ax)
ph.save(fig, 'qc_steadiness.pdf', PLOT_DIR); plt.show()
unsteady.head(20)

### Anything that did not run
Out-of-memory and analytically infeasible configurations are results, not failures; genuine errors are listed too.

In [ ]:
ph.status_report(df)

### Per-architecture method tables

In [ ]:
tables = {}
for arch in ARCHS:
    tables[arch] = ph.method_table(df, arch)
    print(f"\n=== {ph.ARCH_TITLES.get(arch, arch)} ===")
    display(tables[arch])
    tables[arch].to_csv(f'{PLOT_DIR}/table_methods_{arch}.csv', index=False) if __import__('os').makedirs(PLOT_DIR, exist_ok=True) is None else None
    open(f'{PLOT_DIR}/table_methods_{arch}.tex', 'w').write(tables[arch].to_latex(index=False, na_rep='--', float_format='%.3g'))

### Step time and peak memory, per architecture

In [ ]:
for value, fname in [('step_ms', 'methods_step_time'), ('peak_mb', 'methods_peak_memory'), ('overhead_mb', 'methods_memory_overhead')]:
    fig, axes = plt.subplots(1, len(ARCHS), figsize=(5.2 * len(ARCHS), 6), squeeze=False)
    for ax, arch in zip(axes[0], ARCHS):
        ph.plot_method_bars(df, arch, ax, value=value)
    fig.tight_layout(); ph.save(fig, f'{fname}.pdf', PLOT_DIR); plt.show()

### Cross-architecture summary
Step time relative to Adam and peak memory relative to SGD, same architecture and batch size.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
for ax, (value, title) in zip(axes, [('rel_time', 'Step time / Adam'), ('rel_mem', 'Peak memory / SGD')]):
    im = ph.plot_heatmap(df, ax, value=value); ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.8, label='log10')
fig.tight_layout(); ph.save(fig, 'summary_heatmaps.pdf', PLOT_DIR); plt.show()

### Where a Sven step spends its time
Solid: capture (`loss_and_grad`: forward, Jacobian or Gram). Hatched: solve and apply (`optimizer.step`).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
ph.plot_phase_bars(df, ax)
fig.tight_layout(); ph.save(fig, 'sven_phase_breakdown.pdf', PLOT_DIR); plt.show()